# Logging my journey learning LLMs and PyTorch

It is long overdue to start learning PyTorch, and there is no better way to do it than by building a toy LLM. I've gone through a list of papers that helped me to understand how LLMs come to be, and I think it's a good idea to build a toy LLM to better connect the theory with practice.

# Starting the local colab docker image

Use the following command:
```
docker run -it --gpus=all -p 127.0.0.1:9000:8080 --mount type=bind,src="${HOME}/colab/checkpoints",dst=/content/checkpoints us-docker.pkg.dev/colab-images/public/runtime
```

In [59]:
!pwd && ls

/content
checkpoints  sample_data  tiktoken


# Setup PyTorch

In [60]:
import torch

In [61]:
print(f"PyTorch Version: {torch.__version__}")
if (torch.cuda.is_available()):
    device = torch.device("cuda")
    print(f"Using GPU, CUDA version: {torch.version.cuda}, Number of GPUs: {torch.cuda.device_count()}")
else:
    device = torch.device("cpu")

PyTorch Version: 2.10.0+cu128
Using GPU, CUDA version: 12.8, Number of GPUs: 1


# Get a working tokenizer
Let's use OpenAI's tiktoken tokenizer - alghough I am actually interested in building a minimum tokenizer from scratch later.

In [62]:
!git clone https://github.com/openai/tiktoken.git

fatal: destination path 'tiktoken' already exists and is not an empty directory.


In [63]:
import tiktoken

In [64]:
enc = tiktoken.get_encoding("r50k_base")

In [65]:
enc.encode("monkey d luffy")

[49572, 288, 300, 15352]

In [66]:
enc.decode(enc.encode("monkey d luffy"))

'monkey d luffy'

# Get the WebText dataset

In [67]:
from datasets import load_dataset

In [68]:
openwebtext = load_dataset("openwebtext")

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/80 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/80 [00:00<?, ?it/s]

In [69]:
openwebtext

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 8013769
    })
})

In [70]:
openwebtext['train']

Dataset({
    features: ['text'],
    num_rows: 8013769
})

## Test tokenizing the training data

In [71]:
enc.encode(openwebtext['train'][42]['text'])

[5159,
 8305,
 370,
 868,
 510,
 1165,
 1903,
 290,
 1719,
 2761,
 25446,
 736,
 284,
 3993,
 743,
 423,
 257,
 4633,
 2928,
 319,
 262,
 2612,
 11,
 257,
 2050,
 2523,
 198,
 198,
 8061,
 508,
 423,
 5876,
 38193,
 572,
 284,
 3993,
 743,
 307,
 379,
 3220,
 2526,
 286,
 2612,
 5287,
 11,
 4837,
 910,
 13,
 198,
 198,
 464,
 2050,
 11,
 3199,
 287,
 262,
 3427,
 8894,
 4913,
 11,
 3940,
 517,
 621,
 2026,
 11,
 830,
 661,
 329,
 1367,
 812,
 13,
 198,
 198,
 29193,
 1043,
 883,
 508,
 6989,
 1811,
 12513,
 286,
 3595,
 3993,
 547,
 517,
 1884,
 284,
 1205,
 262,
 4006,
 11,
 287,
 543,
 262,
 2612,
 10143,
 284,
 8901,
 6105,
 13,
 198,
 198,
 38897,
 910,
 2252,
 2267,
 318,
 2622,
 284,
 766,
 611,
 257,
 3092,
 286,
 3993,
 5640,
 2612,
 5287,
 393,
 262,
 2792,
 318,
 517,
 3716,
 13,
 198,
 198,
 1,
 42332,
 867,
 286,
 262,
 1243,
 326,
 4646,
 262,
 2863,
 286,
 2612,
 5287,
 635,
 4646,
 47104,
 26,
 922,
 5496,
 11,
 5517,
 11,
 3463,
 2994,
 290,
 407,
 9216,
 1583,
 5045,
 

# Embedding layer

In [72]:
vocabulary_size = enc.n_vocab
d_model = 768

embedding_layer = torch.nn.Embedding(vocabulary_size, d_model)

Here we encode the phrase "monkey d luffy", which is then fed into the embedding_layer,
which produces d_model=768 embeddings, one 768 dimentional vector per token

In [73]:
embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

tensor([[[-0.5496, -0.9280,  0.8916,  ..., -0.3626,  1.0670, -0.8755],
         [ 0.6194,  0.5180,  0.5033,  ...,  1.3957,  0.2241,  0.0966],
         [-0.9937,  0.2419,  1.0564,  ..., -0.1679,  1.7009,  1.2111],
         [-0.2263, -0.3086,  0.5513,  ..., -1.0672, -0.5197, -0.2367]]],
       grad_fn=<EmbeddingBackward0>)

In [74]:
embedding_layer.weight.shape

torch.Size([50257, 768])

Here we can see that the embedding layer is rather simple - it is a n_vocab x d_model matrix. So the embedding process is simply taking w[encoding].

## Positional Encoding

In [75]:
class LearnablePositionalEncoding(torch.nn.Module):
    def __init__(self, max_seq_len, d_model):
        super().__init__()
        self.pos_embeddings = torch.nn.Embedding(max_seq_len, d_model)
    def forward(self, x):
        # x is of shape (batch_size, seq_len, d_model)
        seq_len = x.size(1)

        position_ids = torch.arange(seq_len, dtype=torch.long, device=x.device)

        pos_enc = self.pos_embeddings(position_ids)
        return x + pos_enc


In [76]:
pos_embed_layer = LearnablePositionalEncoding(1024, d_model)

In [77]:
pos_embed_layer(embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")])))

tensor([[[ 0.5875, -1.1507, -0.4461,  ...,  0.9398,  0.4821, -0.3048],
         [ 2.6529, -0.7952,  2.1611,  ...,  0.3477, -0.3808, -0.0286],
         [-3.7330,  0.9582,  0.6319,  ..., -0.8920,  1.1836,  1.5280],
         [-1.1914, -2.4154,  0.5907,  ..., -1.5496, -1.1631, -0.6883]]],
       grad_fn=<AddBackward0>)

# Decoder Only Transformer Layer

This is the key part of the transformer architecture. Each attention layer is made of multi-head attention, normalization, and position-wise feedforward.

In GPT-2, normalization is moved at the beginning of each layer.

In [78]:
embedding = embedding_layer(torch.LongTensor([enc.encode("monkey d luffy")]))

## Layer Normalization

https://arxiv.org/pdf/1607.06450

In [79]:
layer_norm = torch.nn.LayerNorm(d_model)

In [80]:
x = layer_norm(embedding)

In [81]:
x

tensor([[[-0.4676, -0.8470,  0.9778,  ..., -0.2800,  1.1536, -0.7944],
         [ 0.6267,  0.5255,  0.5108,  ...,  1.4014,  0.2321,  0.1049],
         [-1.0052,  0.2282,  1.0412,  ..., -0.1809,  1.6845,  1.1955],
         [-0.1987, -0.2818,  0.5864,  ..., -1.0477, -0.4949, -0.2092]]],
       grad_fn=<NativeLayerNormBackward0>)

In [82]:
x.shape

torch.Size([1, 4, 768])

## Multi-Head Attention

In [83]:
n_head = 12

# project into Q, K, V
projection = torch.nn.Linear(d_model, d_model * 3)

In [84]:
projection(x).shape

torch.Size([1, 4, 2304])

In [85]:
proj = projection(x).view(1, -1, 12, d_model // n_head * 3).permute(0, 2, 1, 3)

In [86]:
q, k, v = proj.chunk(3, dim=3)

In [87]:
import math
atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_model)

In [88]:
mask = torch.tril(torch.ones_like(atten[0][0]))

In [89]:
atten = atten.masked_fill(mask == 0, -float('inf'))

In [90]:
atten

tensor([[[[-0.0559,    -inf,    -inf,    -inf],
          [ 0.0836, -0.0063,    -inf,    -inf],
          [ 0.2341,  0.0580, -0.0788,    -inf],
          [ 0.0411,  0.2520,  0.0424, -0.0956]],

         [[-0.0963,    -inf,    -inf,    -inf],
          [ 0.0457,  0.0591,    -inf,    -inf],
          [-0.0218, -0.0129,  0.0463,    -inf],
          [-0.0242,  0.0332,  0.0437, -0.0413]],

         [[ 0.1307,    -inf,    -inf,    -inf],
          [-0.0048,  0.1561,    -inf,    -inf],
          [-0.0173,  0.0997, -0.0149,    -inf],
          [ 0.0425,  0.0381, -0.2400,  0.0400]],

         [[-0.1594,    -inf,    -inf,    -inf],
          [-0.0587,  0.0265,    -inf,    -inf],
          [ 0.1423, -0.0331, -0.0086,    -inf],
          [-0.0147,  0.0617, -0.0047, -0.0239]],

         [[ 0.0946,    -inf,    -inf,    -inf],
          [ 0.0671, -0.0528,    -inf,    -inf],
          [-0.0564, -0.0157,  0.1118,    -inf],
          [ 0.1204, -0.0934,  0.0513, -0.1308]],

         [[ 0.1416,    -inf,  

In [91]:
softmax = torch.nn.Softmax(dim=-1)

atten = softmax(atten)

In [92]:
atten

tensor([[[[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5225, 0.4775, 0.0000, 0.0000],
          [0.3891, 0.3263, 0.2846, 0.0000],
          [0.2434, 0.3006, 0.2437, 0.2123]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4966, 0.5034, 0.0000, 0.0000],
          [0.3247, 0.3276, 0.3476, 0.0000],
          [0.2432, 0.2575, 0.2603, 0.2391]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4599, 0.5401, 0.0000, 0.0000],
          [0.3198, 0.3595, 0.3206, 0.0000],
          [0.2669, 0.2657, 0.2012, 0.2662]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4787, 0.5213, 0.0000, 0.0000],
          [0.3705, 0.3109, 0.3186, 0.0000],
          [0.2451, 0.2645, 0.2475, 0.2428]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.5300, 0.4700, 0.0000, 0.0000],
          [0.3101, 0.3230, 0.3669, 0.0000],
          [0.2842, 0.2295, 0.2652, 0.2211]],

         [[1.0000, 0.0000, 0.0000, 0.0000],
          [0.4797, 0.5203, 0.0000, 0.0000],
          [0.3332, 0.3

In [93]:
torch.matmul(atten, v).transpose(1, 2).reshape(1, 4, d_model)

tensor([[[-0.2319,  0.7190, -0.2270,  ..., -0.0339,  0.7848, -0.6921],
         [ 0.1953,  0.5486, -0.2154,  ..., -0.1543,  0.2908, -0.4630],
         [ 0.1876,  0.5402, -0.2009,  ..., -0.1795,  0.1488, -0.3747],
         [ 0.3143,  0.3823, -0.2778,  ..., -0.1707,  0.0880, -0.4551]]],
       grad_fn=<UnsafeViewBackward0>)

### Putting it together

In [94]:
import torch.nn.functional as F
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads
        # project into K Q V
        self.input_linear = torch.nn.Linear(d_model, d_model * 3)
    def forward(self, x):
        batch_size, sequence_length, _ = x.size()
        proj = self.input_linear(x).view(batch_size, sequence_length, self.n_heads, self.d_model // self.n_heads * 3).permute(0, 2, 1, 3)
        q, k, v = proj.chunk(3, dim=-1)

        atten = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_model)
        mask = torch.tril(torch.ones_like(atten[0][0]))
        atten = atten.masked_fill(mask == 0, -float('inf'))
        atten = F.softmax(atten, dim=-1)

        return torch.matmul(atten, v).transpose(1, 2).reshape(batch_size, sequence_length, d_model)


In [95]:
multi_head_attention = MultiHeadAttention(d_model, n_head)
multi_head_attention(x)

tensor([[[-0.9063, -0.5025, -0.5141,  ...,  1.3953, -0.3048,  0.1903],
         [-0.6774, -0.4794, -0.1324,  ...,  0.3219, -0.1199,  0.0289],
         [-0.4530, -0.4507, -0.0562,  ..., -0.0704, -0.2765,  0.1227],
         [-0.0636, -0.4256,  0.0483,  ...,  0.1867, -0.1473, -0.0071]]],
       grad_fn=<UnsafeViewBackward0>)

## Position-wise Feed Forward

In [96]:
d_hidden = 3072

In [97]:
class DenseFeedForward(torch.nn.Module):
    def __init__(self, d_model, d_hidden):
        super().__init__()
        self.model = torch.nn.Sequential(
            torch.nn.Linear(d_model, d_hidden),
            torch.nn.ReLU(),
            torch.nn.Linear(d_hidden, d_model)
        )
    def forward(self, x):
        return self.model(x)

In [98]:
y = multi_head_attention(x)

In [99]:
dff = DenseFeedForward(d_model, d_hidden)

In [100]:
dff(y)

tensor([[[-0.0042, -0.0056, -0.0323,  ...,  0.1385,  0.1050, -0.1466],
         [-0.0384,  0.0863, -0.0201,  ...,  0.0772, -0.0490, -0.1415],
         [-0.1176,  0.1024, -0.0024,  ...,  0.1206,  0.0276, -0.1424],
         [-0.1288,  0.0562, -0.0528,  ...,  0.0769,  0.0205, -0.1089]]],
       grad_fn=<ViewBackward0>)

## The Full Transformer Layer

In [101]:
class TransformerDecoderOnly(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head):
        super().__init__()
        self.layer_norm_0 = torch.nn.LayerNorm(d_model)
        self.multi_head_attention = MultiHeadAttention(d_model, n_head)
        self.layer_norm_1 = torch.nn.LayerNorm(d_model)
        self.dff = DenseFeedForward(d_model, d_hidden)
    def forward(self, x):
        # layer normalization first
        x = self.layer_norm_0(x)

        # multi-head attention and residual
        identity = x
        x = self.multi_head_attention(x)
        x = x + identity

        # layer normalization before dense feed forward
        x = self.layer_norm_1(x)

        # dense feed forward and residual
        identity = x
        x = self.dff(x)
        return x + identity

In [102]:
transformer_decoder_only = TransformerDecoderOnly(d_model, d_hidden, n_head)

transformer_decoder_only(embedding)

tensor([[[-0.4139, -0.9260,  0.6421,  ..., -0.0725,  0.8634, -0.7689],
         [ 0.2243,  0.1058,  0.2323,  ...,  1.9642, -0.0454,  0.0919],
         [-1.2442,  0.1205,  0.9572,  ...,  0.2242,  1.3119,  1.3496],
         [-0.5615, -0.4994,  0.7189,  ..., -0.4754, -1.4362, -0.0033]]],
       grad_fn=<AddBackward0>)

# Output Layer

The output layer is rather simple. matrix multiply with the input embedding matrix, and soft max. We can then reverse the tokenization.

In [103]:
torch.matmul(transformer_decoder_only(embedding), embedding_layer.weight.transpose(0, 1))

tensor([[[ 48.9907, -26.0383, -12.2343,  ..., -37.8845,  29.9925,   3.2876],
         [ 15.7032,  56.8231, -26.0290,  ..., -52.1396,  22.4060,  38.5073],
         [ 21.5385,  37.7344, -17.8423,  ...,  13.0473,  19.2213, -18.7747],
         [  2.0395,  20.6598, -26.2742,  ...,  30.3859,   2.1691, -24.6494]]],
       grad_fn=<UnsafeViewBackward0>)

In [104]:
class Output(torch.nn.Module):
    def __init__(self, embedding_layer_weights):
        super().__init__()
        # Removed the extra LayerNorm here as it can skew initial logits
        self.embedding_layer_weights = embedding_layer_weights

    def forward(self, x):
        # x: (batch, seq, d_model)
        # weights: (vocab, d_model)
        return torch.matmul(x, self.embedding_layer_weights.transpose(0, 1))

# The full model

In [105]:
class ToyGPT(torch.nn.Module):
    def __init__(self, d_model, d_hidden, n_head, vocab_size, layers, max_seq_len):
        super().__init__()
        self.embedding_layer = torch.nn.Embedding(vocab_size, d_model)
        self.pos_encoding_layer = LearnablePositionalEncoding(max_seq_len, d_model)
        self.transformers = torch.nn.Sequential(*[TransformerDecoderOnly(d_model, d_hidden, n_head) for _ in range(layers)])
        # Standard GPT-2: Final LayerNorm before the output head
        self.ln_f = torch.nn.LayerNorm(d_model)
        self.output_layer = Output(self.embedding_layer.weight)

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, torch.nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, torch.nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        x = self.embedding_layer(x)
        x = self.pos_encoding_layer(x)
        x = self.transformers(x)
        x = self.ln_f(x) # Apply the missing final normalization
        return self.output_layer(x)

# Training

In [106]:
batch_size = 48
max_seq_len = 1024

In [107]:
enc._special_tokens

{'<|endoftext|>': 50256}

Pre-tokenize the training data to avoid CPU bottleneck.

In [108]:
from datasets import Dataset

rows = openwebtext["train"].num_rows
bos_token = '<|endoftext|>'

def tokenize(examples):
    out = enc.encode(examples['text']+ bos_token, allowed_special={bos_token})
    return {'tokenized_text': out}

tokenized_data = openwebtext["train"].map(tokenize, remove_columns=openwebtext["train"].column_names, num_proc=32)


In [109]:
def do_checkpoint(model, optimizer, running_encoding, row_number):
    torch.save({
        'row_number': row_number,
        'running_encoding': running_encoding,
        'optimizer_state_dict': optimizer.state_dict(),
        'model_state_dict': model.state_dict(),
    }, 'checkpoints/check_point_{}'.format(row_number))

In [110]:
import gc
from tqdm.notebook import tqdm

def train(start_row_number = 0, check_point_interval = 100000, summary_writer = None):
    tokens_per_batch = (batch_size + 1) * max_seq_len // 2

    encoding = []

    model = ToyGPT(d_model, d_hidden, n_head, enc.n_vocab, 12, max_seq_len)
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0006, weight_decay=0.01)
    model.to(device)
    model.train()

    model = torch.compile(model)

    if start_row_number != 0:
        checkpoint = torch.load('checkpoints/check_point_{}'.format(start_row_number))

        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

        encoding = checkpoint['running_encoding']

    write_summary_threshold = start_row_number
    check_point_threshold = (start_row_number + check_point_interval) // check_point_interval * check_point_interval

    data = tokenized_data.select(range(start_row_number, tokenized_data.num_rows)).to_iterable_dataset(num_shards=8)


    for row in enumerate(tqdm(data, total = rows, initial = start_row_number)):
        encoding.extend([enc._special_tokens[bos_token]] + row[1]['tokenized_text'])
        cur_row_number = start_row_number + row[0]
        if len(encoding) > tokens_per_batch:
            residual = encoding[tokens_per_batch:]
            encoding = encoding[0:tokens_per_batch + 1]
            batch = None
            target = None
            for i in range(0, batch_size):
              batch_row = torch.LongTensor(encoding[i * max_seq_len // 2 : i * max_seq_len // 2 + max_seq_len])
              batch_row = batch_row.to(device)
              target_row = torch.LongTensor(encoding[i * max_seq_len // 2 + 1 : i * max_seq_len // 2 + max_seq_len + 1])
              target_row = target_row.to(device)
              if batch is None:
                  batch = batch_row
                  target = target_row
              else:
                  batch = torch.cat((batch, batch_row), dim=0)
                  target = torch.cat((target, target_row), dim=0)

            batch = batch.to(device)
            batch = batch.reshape(batch_size, max_seq_len)
            target = target.to(device)
            target = target.reshape(batch_size, max_seq_len)

            optimizer.zero_grad()

            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                out = model(batch)
                loss = F.cross_entropy(out.view(-1, enc.n_vocab), target.view(-1))

            loss.backward()
            optimizer.step()

            encoding = residual
            if summary_writer is not None and cur_row_number >= write_summary_threshold:
                write_summary_threshold = (write_summary_threshold + 1000) // 1000 * 1000
                summary_writer.add_scalar('Loss/train', loss.item(), cur_row_number)

            if (cur_row_number + 1 >= check_point_threshold):
                print("check pointing at {}".format(cur_row_number + 1))
                do_checkpoint(model, optimizer, encoding, cur_row_number + 1)
                check_point_threshold = (cur_row_number + 1 + check_point_interval) // check_point_interval * check_point_interval

            gc.collect()

### Validation: Architecture and Initial Loss
For a randomly initialized model, the expected cross-entropy loss is $-\ln(1/V) = \ln(V)$, where $V$ is the vocabulary size. Let's verify this and check the output shapes.

In [111]:
import math

# 1. Check Output Shapes
test_model = ToyGPT(d_model, d_hidden, n_head, vocabulary_size, layers=1, max_seq_len=max_seq_len)
test_input = torch.randint(0, vocabulary_size, (1, 8)) # Batch size 1, Seq len 8
with torch.no_grad():
    logits = test_model(test_input)

print(f"Input shape: {test_input.shape}")
print(f"Output (logits) shape: {logits.shape} (Expected: [1, 8, {vocabulary_size}])")

# 2. Check Initial Loss
targets = torch.randint(0, vocabulary_size, (1, 8))
loss = F.cross_entropy(logits.view(-1, vocabulary_size), targets.view(-1))

theoretical_loss = math.log(vocabulary_size)
print(f"\nTheoretical Initial Loss: {theoretical_loss:.4f}")
print(f"Actual Initial Loss:      {loss.item():.4f}")
print(f"Difference:              {abs(loss.item() - theoretical_loss):.4f}")

if abs(loss.item() - theoretical_loss) < 0.5:
    print("\n✅ The initial loss is consistent with random initialization.")
else:
    print("\n⚠️ Initial loss is significantly different from theory. Check initialization or scaling.")

Input shape: torch.Size([1, 8])
Output (logits) shape: torch.Size([1, 8, 50257]) (Expected: [1, 8, 50257])

Theoretical Initial Loss: 10.8249
Actual Initial Loss:      11.2887
Difference:              0.4638

✅ The initial loss is consistent with random initialization.


In [112]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [113]:
from torch.utils.tensorboard import SummaryWriter

summary_writer = SummaryWriter('./checkpoints/adamw_run_lr_0006')

In [114]:
%tensorboard --logdir checkpoints/adamw_run_lr_0006 --bind_all


<IPython.core.display.Javascript object>

In [ ]:
train(start_row_number=0, check_point_interval=100000, summary_writer = summary_writer)

  0%|          | 0/8013769 [00:00<?, ?it/s]